## Environment Setup

In [ ]:
import subprocess, sys
PKGS = ['pyspark','matplotlib','seaborn','networkx','pandas','numpy',
        'scipy','wordcloud','openpyxl']
for p in PKGS:
    subprocess.check_call([sys.executable,'-m','pip','install',p,'-q'])
print('All packages ready.')

In [ ]:
import os, time, warnings, math
from itertools import combinations
from collections import defaultdict
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
import networkx as nx
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist
try:
    from wordcloud import WordCloud
    HAS_WC = True
except ImportError:
    HAS_WC = False

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

DARK_BG   = '#0d1117'
CARD_BG   = '#161b22'
ACCENT1   = '#f9c74f'
ACCENT2   = '#90e0ef'
ACCENT3   = '#f94144'
ACCENT4   = '#43aa8b'
ACCENT5   = '#c77dff'
PAL       = [ACCENT1, ACCENT2, ACCENT3, ACCENT4, ACCENT5,
             '#f77f00','#2d6a4f','#e9c46a','#264653','#e76f51']

plt.rcParams.update({
    'figure.facecolor' : DARK_BG,
    'axes.facecolor'   : CARD_BG,
    'axes.edgecolor'   : '#30363d',
    'axes.labelcolor'  : '#c9d1d9',
    'axes.titlecolor'  : '#f0f6fc',
    'axes.grid'        : True,
    'grid.color'       : '#21262d',
    'grid.linewidth'   : 0.8,
    'xtick.color'      : '#8b949e',
    'ytick.color'      : '#8b949e',
    'text.color'       : '#c9d1d9',
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 10.5,
    'legend.facecolor' : '#21262d',
    'legend.edgecolor' : '#30363d',
    'figure.dpi'       : 120,
})


spark = (
    SparkSession.builder
    .appName('MBA_APriori_PCY')
    .config('spark.driver.memory','4g')
    .config('spark.sql.shuffle.partitions','8')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark {spark.version} ready.')

## Data Loading

In [ ]:
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('data', exist_ok=True)


def generate_synthetic(n_transactions=6000, seed=42):
    rng = np.random.default_rng(seed)

    categories = {
        'dairy': ['whole milk', 'butter', 'cream cheese', 'yogurt', 'whipped cream',
                  'sour cream', 'condensed milk', 'curd'],
        'produce': ['other vegetables', 'root vegetables', 'tropical fruit', 'citrus fruit',
                    'pip fruit', 'onions', 'berries', 'herbs', 'specialty vegetables'],
        'bakery': ['rolls/buns', 'brown bread', 'pastry', 'waffles', 'cake bar', 'cookies'],
        'beverages': ['soda', 'bottled water', 'fruit juice', 'bottled beer',
                      'canned beer', 'coffee', 'tea', 'misc beverages'],
        'meat': ['sausage', 'pork', 'beef', 'chicken', 'frankfurter', 'ham', 'turkey'],
        'snacks': ['chocolate', 'nuts', 'candy bars', 'specialty bar', 'chips', 'popcorn'],
        'household': ['dish cleaner', 'detergent', 'fabric softener', 'toilet cleaner',
                      'kitchen towels', 'napkins', 'light bulbs', 'batteries'],
        'pantry': ['pasta', 'rice', 'salt', 'sugar', 'oil', 'margarine', 'jam', 'ketchup'],
    }
    all_items = [item for cat in categories.values() for item in cat]
    cat_of = {item: cat for cat, items in categories.items() for item in items}

    rows, dates = [], []
    import datetime
    base_date = datetime.date(2023, 1, 1)

    for tid in range(n_transactions):
        basket_size = int(rng.choice([2, 3, 4, 5, 6, 7, 8], p=[0.05, 0.15, 0.25, 0.25, 0.15, 0.10, 0.05]))
        # Pick a seed category; sample mostly within it
        seed_cat = rng.choice(list(categories.keys()))
        seed_items = categories[seed_cat]
        n_from_cat = min(basket_size, rng.integers(1, len(seed_items) + 1))
        basket = set(rng.choice(seed_items, size=n_from_cat, replace=False).tolist())
        # Fill remaining slots from global pool
        remaining = [i for i in all_items if i not in basket]
        extra_n = max(0, basket_size - len(basket))
        if extra_n > 0 and remaining:
            basket.update(rng.choice(remaining,
                                     size=min(extra_n, len(remaining)),
                                     replace=False).tolist())
        day_offset = int(rng.integers(0, 365))
        date_str = (base_date + datetime.timedelta(days=day_offset)).isoformat()
        for item in basket:
            rows.append({'transaction_id': str(tid), 'item': item,
                         'date': date_str, 'category': cat_of[item]})

    return pd.DataFrame(rows)


# Load data
GROCERIES_PATH = 'data/Groceries_dataset.csv'
ONLINE_RETAIL_PATH = 'data/Online Retail.xlsx'

if os.path.exists(GROCERIES_PATH):
    raw_pdf = pd.read_csv(GROCERIES_PATH)
    raw_pdf.columns = ['transaction_id', 'date', 'item']
    raw_pdf['transaction_id'] = raw_pdf['transaction_id'].astype(str)
    raw_pdf['item'] = raw_pdf['item'].str.strip().str.lower()
    raw_pdf['date'] = pd.to_datetime(raw_pdf['date'], errors='coerce')
    raw_pdf['category'] = 'unknown'
    DATASET_NAME = 'Groceries (Kaggle)'
elif os.path.exists(ONLINE_RETAIL_PATH):
    tmp = pd.read_excel(ONLINE_RETAIL_PATH, engine='openpyxl')
    tmp = tmp[['InvoiceNo', 'InvoiceDate', 'Description']].dropna()
    tmp.columns = ['transaction_id', 'date', 'item']
    tmp = tmp[~tmp['transaction_id'].astype(str).str.startswith('C')]
    raw_pdf = tmp.copy()
    raw_pdf['item'] = raw_pdf['item'].str.strip().str.lower()
    raw_pdf['transaction_id'] = raw_pdf['transaction_id'].astype(str)
    raw_pdf['date'] = pd.to_datetime(raw_pdf['date'], errors='coerce')
    raw_pdf['category'] = 'unknown'
    DATASET_NAME = 'Online Retail (UCI)'
else:
    print('No dataset found — generating synthetic data...')
    raw_pdf = generate_synthetic(n_transactions=6000)
    raw_pdf['date'] = pd.to_datetime(raw_pdf['date'])
    DATASET_NAME = 'Synthetic Grocery'

raw_pdf = raw_pdf.dropna(subset=['item'])
raw_pdf = raw_pdf[raw_pdf['item'].str.strip() != '']

# Spark DF
transactions_df = spark.createDataFrame(
    raw_pdf[['transaction_id', 'item']].astype(str)
).cache()

n_transactions = transactions_df.select('transaction_id').distinct().count()
n_unique_items = transactions_df.select('item').distinct().count()
basket_sizes_pdf = (transactions_df.groupBy('transaction_id').count().toPandas())
avg_basket = basket_sizes_pdf['count'].mean()

print(f'Dataset      : {DATASET_NAME}')
print(f'Transactions : {n_transactions:,}')
print(f'Unique items : {n_unique_items:,}')
print(f'Avg basket   : {avg_basket:.2f}')